In [103]:
import pandas as pd
import numpy as np
import networkx as nx
from pathlib import Path

In [101]:
# Get the notebook's directory and go up to project root
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
data_folder = project_root / "data" / "resources"
PilotDC9 = data_folder / "PilotStudy_Afekta"

In [102]:
graph_gml = data_folder / "generated" / "modified_graph.gml"

In [104]:
G = nx.read_gml(graph_gml)

In [25]:
chebi_match = pd.read_csv(PilotDC9 / "AFEKTA_chebis.csv")

In [68]:
name_match = pd.read_excel(PilotDC9 / "Supplementary_Material_2_Results_Dennisse_Avella.xlsx",sheet_name="keyIDs")

In [100]:
mask = name_match["Idlevel"] <= 2

# Clean empty values or '-' to None
curated = (
    name_match.loc[mask, "CuratedID"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
name_match.loc[mask, "CuratedID"] = curated.where(curated.notna(), None)
# Clean chebi_match
chebi_clean = chebi_match.copy()
chebi_clean["Input name"] = (
    chebi_clean["Input name"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "-": pd.NA})
)
chebi_clean["CHEBI_ID"] = chebi_clean["CHEBI_ID"].replace({"": None, "-": None})
# Create lookup table from chebi_clean
lookup = (
    chebi_clean.dropna(subset=["Input name"])
    .drop_duplicates(subset=["Input name"], keep="first")
    .set_index("Input name")["CHEBI_ID"]
)
# Map Chebi_IDs to name_match
name_match.loc[mask, "Chebi_ID"] = name_match.loc[mask, "CuratedID"].map(lookup)

In [129]:
name_match[name_match['CuratedID'] == 'Diethanolamine']

,Class,CuratedID,Idlevel,DatabaseID,Chebi_ID
34,Other compounds,Diethanolamine,1.0,HMDB0004437,28123
